# 📖 Notebook 3: NoSQL Data Models

Relational databases are the default, but sometimes your data doesn't fit neatly into tables. This notebook explores **alternative data models** — document stores, key-value patterns, and wide-column designs.

We'll use PostgreSQL's built-in **JSONB** support to simulate document-style modeling without installing MongoDB. This lets you compare relational and document approaches side by side in the same database.

## Learning Objectives

By the end of this notebook, you'll understand:
- When to choose document, key-value, or wide-column models
- How document-style data (JSON) differs from normalized tables
- The embedding vs referencing trade-off
- How to model key-value and wide-column patterns
- Why relational is still the default for most interview problems

## 🛠️ Setup

```bash
cd 01-foundations/data-modeling
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import json
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "data_modeling_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def query(sql, params=None):
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cursor.execute(sql, params)
    rows = cursor.fetchall()
    conn.close()
    return rows

def execute(sql, params=None):
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute(sql, params)
    conn.commit()
    conn.close()

try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ Connection failed: {e}")

## 🗺️ When to Choose Which Database Model

Before diving in, here's the decision framework:

```
┌───────────────────────────────────────────────────────────────┐
│                   Which database model?                       │
├───────────────────────────────────────────────────────────────┤
│                                                               │
│  Need ACID transactions?  ──── YES ──→  SQL (PostgreSQL)      │
│         │                                                     │
│         NO                                                    │
│         │                                                     │
│  Flexible/nested schema?  ──── YES ──→  Document (MongoDB)    │
│         │                                                     │
│         NO                                                    │
│         │                                                     │
│  Simple key→value lookup? ──── YES ──→  Key-Value (Redis)     │
│         │                                                     │
│         NO                                                    │
│         │                                                     │
│  Massive write volume?    ──── YES ──→  Wide-Column (Cassandra)│
│         │                                                     │
│         NO                                                    │
│         │                                                     │
│  Use SQL (PostgreSQL) — it's the safe default                 │
│                                                               │
└───────────────────────────────────────────────────────────────┘
```

**Interview tip:** Most of the time, the right answer is SQL. Only choose NoSQL when your requirements clearly demand it.

## 📄 Document Model: Embedding vs Referencing

Document databases (like MongoDB) store data as JSON-like documents. The key decision is:

- **Embed** related data inside the document (denormalize)
- **Reference** related data in separate documents (normalize)

PostgreSQL has excellent JSONB support, so we can explore both approaches right here.

In [ ]:
# Create a document-style table using JSONB
execute("""
    CREATE TABLE IF NOT EXISTS user_profiles_doc (
        id TEXT PRIMARY KEY,
        data JSONB NOT NULL
    );
""")

# Build embedded documents from our normalized data
# Each user document contains their posts, with comments and likes nested inside

users = query("SELECT id, username, email, display_name, bio FROM users WHERE id <= 5")

for user in users:
    uid = user['id']
    
    # Get this user's posts with embedded comments and like counts
    posts = query("""
        SELECT p.id, p.content, p.created_at::text as created_at,
               COALESCE(json_agg(
                   json_build_object('username', cu.username, 'content', c.content)
               ) FILTER (WHERE c.id IS NOT NULL), '[]') AS comments,
               COUNT(DISTINCT l.id) AS like_count
        FROM posts p
        LEFT JOIN comments c ON c.post_id = p.id
        LEFT JOIN users cu ON c.user_id = cu.id
        LEFT JOIN likes l ON l.post_id = p.id
        WHERE p.user_id = %s
        GROUP BY p.id, p.content, p.created_at
        ORDER BY p.created_at DESC
        LIMIT 5
    """, (uid,))
    
    # Build the full document
    doc = {
        "username": user['username'],
        "email": user['email'],
        "display_name": user['display_name'],
        "bio": user['bio'],
        "posts": [
            {
                "content": p['content'],
                "created_at": p['created_at'],
                "like_count": p['like_count'],
                "comments": p['comments'] if isinstance(p['comments'], list) else json.loads(p['comments'])
            }
            for p in posts
        ]
    }
    
    execute(
        "INSERT INTO user_profiles_doc (id, data) VALUES (%s, %s) ON CONFLICT (id) DO UPDATE SET data = %s",
        (f"user:{uid}", json.dumps(doc), json.dumps(doc))
    )

print("✅ Created document-style user profiles")

# Show one document
doc_row = query("SELECT data FROM user_profiles_doc WHERE id = 'user:1'")[0]
print()
print("📄 Document for user:1 (embedded model):")
print(json.dumps(doc_row['data'], indent=2)[:800] + "\n  ...")

In [ ]:
# Compare: reading a user profile with posts

print("⏱️ Read Performance: Relational vs Document")
print("=" * 55)

# Relational: requires JOIN across multiple tables
times_rel = []
for _ in range(100):
    start = time.time()
    query("""
        SELECT u.username, u.email, u.bio,
               p.content, p.created_at,
               COUNT(DISTINCT l.id) AS like_count
        FROM users u
        LEFT JOIN posts p ON p.user_id = u.id
        LEFT JOIN likes l ON l.post_id = p.id
        WHERE u.id = 1
        GROUP BY u.id, p.id
        ORDER BY p.created_at DESC
        LIMIT 5
    """)
    times_rel.append((time.time() - start) * 1000)

# Document: single lookup, all data embedded
times_doc = []
for _ in range(100):
    start = time.time()
    query("SELECT data FROM user_profiles_doc WHERE id = 'user:1'")
    times_doc.append((time.time() - start) * 1000)

avg_rel = sum(times_rel) / len(times_rel)
avg_doc = sum(times_doc) / len(times_doc)

print(f"  Relational (JOIN):  {avg_rel:.2f} ms avg")
print(f"  Document (JSONB):   {avg_doc:.2f} ms avg")
print(f"  Speedup:            {avg_rel / avg_doc:.1f}×")
print()
print("💡 Document reads are faster because everything is in one place.")
print("   But updating a single comment now means rewriting the whole document.")

## 📄 Embedding vs Referencing Decision

| Factor | Embed | Reference |
|--------|-------|-----------|
| Data read together? | ✅ Always | ❌ Sometimes |
| Data changes independently? | ❌ Rarely | ✅ Often |
| Data is small? | ✅ Yes | ❌ Can be large |
| Need to query nested data? | ❌ Hard | ✅ Easy |

**Embed** when data is read together and rarely changes independently (user + shipping address).  
**Reference** when data changes independently or is shared (post + author profile).

In [ ]:
# Querying JSONB: PostgreSQL lets you query inside JSON documents

print("🔍 Querying inside JSONB documents:")
print("=" * 55)

# Find users by a field inside the JSON
result = query("""
    SELECT id, data->>'username' AS username, data->>'email' AS email
    FROM user_profiles_doc
    WHERE data->>'username' = 'user_1'
""")
print(f"  Find by username: {result}")

# Count posts per user inside the document
post_counts = query("""
    SELECT
        data->>'username' AS username,
        jsonb_array_length(data->'posts') AS post_count
    FROM user_profiles_doc
    ORDER BY jsonb_array_length(data->'posts') DESC
""")
print()
print("  Posts per user (from embedded documents):")
for row in post_counts:
    print(f"    @{row['username']}: {row['post_count']} posts")

print()
print("💡 PostgreSQL JSONB supports operators like ->> (text), -> (json),")
print("   jsonb_array_length(), and even GIN indexes for fast JSON queries.")

## 🔑 Key-Value Model

The simplest data model: look up a value by its key. No joins, no queries, no schema.

Think of it like a giant dictionary/hashmap:
- `session:abc123` → `{user_id: 42, expires: ...}`
- `cache:user:1:profile` → `{username: "alice", ...}`
- `rate_limit:ip:10.0.0.1` → `{count: 47, window_start: ...}`

Used for: caching, sessions, feature flags, rate limiting, counters.

In [ ]:
# Simulate a key-value store in PostgreSQL
execute("""
    CREATE TABLE IF NOT EXISTS kv_store (
        key TEXT PRIMARY KEY,
        value JSONB NOT NULL,
        expires_at TIMESTAMP
    );
""")

# Store some key-value pairs (like you would in Redis)
kv_data = [
    ("session:abc123", {"user_id": 1, "username": "user_1", "role": "admin"}),
    ("session:def456", {"user_id": 2, "username": "user_2", "role": "user"}),
    ("cache:user:1:profile", {"username": "user_1", "display_name": "User 1", "followers": 42}),
    ("rate_limit:ip:192.168.1.1", {"count": 47, "window_start": "2024-01-01T10:00:00"}),
    ("feature_flag:dark_mode", {"enabled": True, "rollout_pct": 25}),
]

for key, value in kv_data:
    execute(
        "INSERT INTO kv_store (key, value) VALUES (%s, %s) ON CONFLICT (key) DO UPDATE SET value = %s",
        (key, json.dumps(value), json.dumps(value))
    )

print("🔑 Key-Value Store Contents:")
print("=" * 60)
rows = query("SELECT key, value FROM kv_store ORDER BY key")
for row in rows:
    print(f"  {row['key']:<35} → {json.dumps(row['value'])}")

print()

# Demonstrate lookup speed
times_kv = []
for _ in range(100):
    start = time.time()
    query("SELECT value FROM kv_store WHERE key = 'session:abc123'")
    times_kv.append((time.time() - start) * 1000)

print(f"⚡ Lookup time: {sum(times_kv)/len(times_kv):.2f} ms avg (100 iterations)")
print()
print("💡 Key-value is the fastest model for exact lookups.")
print("   But you can ONLY look up by key — no 'find all admins' queries.")

## 📊 Wide-Column Model

Wide-column databases (Cassandra, HBase) organize data by **partition key** and **clustering key**.

They're designed for:
- Massive write throughput (append-only)
- Time-series data
- Data that's always queried by the same key

The data model looks like:
```
Partition Key: user_id
Clustering Key: timestamp (sorted)

user_1 | 2024-01-03 | "Posted photo"   | {likes: 5}
user_1 | 2024-01-02 | "Hello world"    | {likes: 3}
user_1 | 2024-01-01 | "First post"     | {likes: 1}
user_2 | 2024-01-03 | "Good morning"   | {likes: 8}
```

All rows with the same partition key are stored together on disk, making range scans fast.

In [ ]:
# Simulate a wide-column model: user activity timeline
execute("""
    CREATE TABLE IF NOT EXISTS user_activity (
        user_id INTEGER NOT NULL,
        event_time TIMESTAMP NOT NULL,
        event_type TEXT NOT NULL,
        event_data JSONB,
        PRIMARY KEY (user_id, event_time)
    );
""")

# Insert activity events (simulating Cassandra-style writes)
execute("""
    INSERT INTO user_activity (user_id, event_time, event_type, event_data)
    SELECT
        (floor(random() * 5) + 1)::int,
        NOW() - (random() * interval '30 days'),
        (ARRAY['post', 'like', 'comment', 'follow', 'login'])[floor(random() * 5 + 1)::int],
        json_build_object(
            'target_id', floor(random() * 100 + 1)::int,
            'ip', '192.168.1.' || floor(random() * 255 + 1)::int
        )::jsonb
    FROM generate_series(1, 100)
    ON CONFLICT DO NOTHING;
""")

print("📊 Wide-Column Style: User Activity Timeline")
print("=" * 65)
print()

# Query pattern 1: Get recent activity for one user (fast — partition key scan)
activity = query("""
    SELECT user_id, event_time, event_type, event_data
    FROM user_activity
    WHERE user_id = 1
    ORDER BY event_time DESC
    LIMIT 5;
""")

print("✅ FAST: Get user 1's recent activity (query by partition key):")
for a in activity:
    print(f"  [{a['event_time']}] {a['event_type']}: {json.dumps(a['event_data'])}")

print()
print("❌ SLOW in Cassandra: Find all 'login' events across ALL users")
print("   This requires scanning every partition — a 'full table scan'.")
print("   In a real wide-column DB, you'd create a second table:")
print("   activity_by_type(event_type, event_time, user_id, data)")
print()
print("💡 Wide-column models require you to design tables around query patterns.")
print("   Each query pattern may need its own table with duplicated data.")

## 🆚 Model Comparison

Let's summarize when each model shines:

In [ ]:
print("🆚 Database Model Comparison:")
print("=" * 75)
print()
print(f"{'Model':<16} {'Best For':<28} {'Trade-off':<30}")
print("-" * 75)
models = [
    ("SQL",          "Most applications",         "Complex JOINs can be slow"),
    ("Document",     "Flexible/nested schemas",   "Hard to query across docs"),
    ("Key-Value",    "Caching, sessions",         "Lookup by key only"),
    ("Wide-Column",  "Time-series, high writes",  "Must design per-query tables"),
    ("Graph",        "Deep relationships",        "Rarely needed, complex ops"),
]

for model, best_for, tradeoff in models:
    print(f"  {model:<14} {best_for:<28} {tradeoff:<30}")

print()
print("📌 Interview default: SQL (PostgreSQL)")
print("   Only switch if your requirements CLEARLY demand another model.")
print("   Even then, you'll often use SQL + a specialized DB together.")

## 🧹 Cleanup

In [ ]:
# Clean up tables we created in this notebook
execute("DROP TABLE IF EXISTS user_profiles_doc")
execute("DROP TABLE IF EXISTS kv_store")
execute("DROP TABLE IF EXISTS user_activity")
print("🧹 Cleaned up temporary tables")

## 📚 Summary

### Key Takeaways

1. **SQL is the default** — most system design problems map naturally to relational tables
2. **Document stores** shine when data is deeply nested or schema evolves rapidly
3. **Key-value stores** are for fast lookups by exact key — caching, sessions, feature flags
4. **Wide-column stores** handle massive write volumes and time-series data
5. **Embed vs reference** is the fundamental document modeling decision
6. **You'll often combine models** — SQL for truth, Redis for caching, etc.

### Interview Tip

> Don't pick exotic databases to impress. Pick PostgreSQL, explain why it fits,  
> and mention alternatives only when requirements clearly call for them:
> *"I'd use Redis here as a cache layer since we need sub-millisecond reads  
> for session lookups, but the source of truth stays in Postgres."*

### Next Up

In **Notebook 4**, we'll explore **schema evolution** — how to change your schema over time without breaking your application.